# Algorithm Comparison Test
## Quick 3-Epoch Accuracy Test

**Purpose**: Compare ResNet18, ResNet50, and MobileNetV2 with 3 epochs

**Note**: This is a separate test notebook that won't affect the working project

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import time
import copy
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


## 1. Load Data

In [2]:
import re

DATASET_PATH = r'c:\Users\vimal\OneDrive\Desktop\FINAL AIML\Waste segregation.v1i.multiclass'
TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
TEST_DIR = os.path.join(DATASET_PATH, 'test')
VALID_DIR = os.path.join(DATASET_PATH, 'valid')

train_csv = pd.read_csv(os.path.join(TRAIN_DIR, '_classes.csv'))
test_csv = pd.read_csv(os.path.join(TEST_DIR, '_classes.csv'))
valid_csv = pd.read_csv(os.path.join(VALID_DIR, '_classes.csv'))

def extract_plastic_type(filename):
    match = re.match(r'^([A-Z]+)', filename)
    if match and match.group(1) in ['HDPE','LDPE','PET','PP','PS','PVC','OTHERS']:
        return match.group(1)
    return 'UNKNOWN'

train_csv['plastic_type'] = train_csv['filename'].apply(extract_plastic_type)
test_csv['plastic_type'] = test_csv['filename'].apply(extract_plastic_type)
valid_csv['plastic_type'] = valid_csv['filename'].apply(extract_plastic_type)

train_csv = train_csv[train_csv['plastic_type'] != 'UNKNOWN'].reset_index(drop=True)
test_csv = test_csv[test_csv['plastic_type'] != 'UNKNOWN'].reset_index(drop=True)
valid_csv = valid_csv[valid_csv['plastic_type'] != 'UNKNOWN'].reset_index(drop=True)

plastic_types = sorted(train_csv['plastic_type'].unique())
class_to_idx = {cls: idx for idx, cls in enumerate(plastic_types)}
idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

train_csv['label'] = train_csv['plastic_type'].map(class_to_idx)
test_csv['label'] = test_csv['plastic_type'].map(class_to_idx)
valid_csv['label'] = valid_csv['plastic_type'].map(class_to_idx)

num_classes = len(plastic_types)
print(f'Classes: {plastic_types}')
print(f'Num classes: {num_classes}')

Classes: ['HDPE', 'LDPE', 'OTHERS', 'PET', 'PP', 'PS']
Num classes: 6


## 2. Data Preparation

In [3]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 3  # Quick test with 3 epochs
LR = 0.001

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

class PlasticDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = csv_file.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = os.path.join(self.img_dir, row['filename'])
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), 'white')
        if self.transform:
            image = self.transform(image)
        return image, row['label']

train_dataset = PlasticDataset(train_csv, TRAIN_DIR, train_transforms)
valid_dataset = PlasticDataset(valid_csv, VALID_DIR, val_transforms)
test_dataset = PlasticDataset(test_csv, TEST_DIR, val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train: {len(train_loader)} batches')
print(f'Valid: {len(valid_loader)} batches')
print(f'Test: {len(test_loader)} batches')

Train: 184 batches
Valid: 47 batches
Test: 24 batches


## 3. Model Architectures

In [4]:
def create_resnet18(num_classes):
    model = models.resnet18(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def create_resnet50(num_classes):
    model = models.resnet50(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def create_mobilenetv2(num_classes):
    model = models.mobilenet_v2(pretrained=True)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

print('Models defined')

Models defined


## 4. Quick Training Function (3 Epochs)

In [5]:
def quick_train(model, train_loader, valid_loader, criterion, optimizer, num_epochs, model_name):
    best_acc = 0.0
    history = {'train_acc': [], 'val_acc': []}
    start_time = time.time()
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        
        # Train
        model.train()
        train_correct = 0
        train_total = 0
        
        for inputs, labels in tqdm(train_loader, desc='Train'):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            loss.backward()
            optimizer.step()
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
        
        train_acc = train_correct / train_total
        history['train_acc'].append(train_acc)
        
        # Validate
        model.eval()
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, labels in tqdm(valid_loader, desc='Valid'):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
        
        val_acc = val_correct / val_total
        history['val_acc'].append(val_acc)
        
        print(f'Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}')
        
        if val_acc > best_acc:
            best_acc = val_acc
    
    time_elapsed = time.time() - start_time
    print(f'\nTime: {time_elapsed//60:.0f}m {time_elapsed%60:.0f}s')
    print(f'Best Val Acc: {best_acc:.4f}')
    
    return model, history, best_acc, time_elapsed

print('Training function ready')

Training function ready


## 5. Train All 3 Models (3 Epochs Each)

In [6]:
results = {}
criterion = nn.CrossEntropyLoss()

print('='*70)
print('QUICK 3-EPOCH COMPARISON TEST')
print('='*70)

QUICK 3-EPOCH COMPARISON TEST


In [7]:
print('\n### MODEL 1: ResNet18 ###')
model1 = create_resnet18(num_classes).to(device)
opt1 = optim.Adam(model1.parameters(), lr=LR)
model1, hist1, acc1, time1 = quick_train(model1, train_loader, valid_loader, criterion, opt1, NUM_EPOCHS, 'ResNet18')
results['ResNet18'] = {'model': model1, 'history': hist1, 'best_acc': acc1, 'time': time1}


### MODEL 1: ResNet18 ###

Epoch 1/3


Valid: 100%|██████████| 47/47 [01:18<00:00,  1.66s/it]


Train Acc: 0.9063 | Val Acc: 0.8591

Epoch 2/3


Valid: 100%|██████████| 47/47 [01:14<00:00,  1.59s/it]


Train Acc: 0.9465 | Val Acc: 0.9376

Epoch 3/3


Valid: 100%|██████████| 47/47 [00:54<00:00,  1.15s/it]

Train Acc: 0.9637 | Val Acc: 0.9819

Time: 33m 49s
Best Val Acc: 0.9819


In [8]:
print('\n### MODEL 2: ResNet50 ###')
model2 = create_resnet50(num_classes).to(device)
opt2 = optim.Adam(model2.parameters(), lr=LR)
model2, hist2, acc2, time2 = quick_train(model2, train_loader, valid_loader, criterion, opt2, NUM_EPOCHS, 'ResNet50')
results['ResNet50'] = {'model': model2, 'history': hist2, 'best_acc': acc2, 'time': time2}


### MODEL 2: ResNet50 ###

Epoch 1/3


Train:   7%|▋         | 12/184 [04:09<59:38, 20.81s/it] 


KeyboardInterrupt: 

In [9]:
print('\n### MODEL 3: MobileNetV2 ###')
model3 = create_mobilenetv2(num_classes).to(device)
opt3 = optim.Adam(model3.parameters(), lr=LR)
model3, hist3, acc3, time3 = quick_train(model3, train_loader, valid_loader, criterion, opt3, NUM_EPOCHS, 'MobileNetV2')
results['MobileNetV2'] = {'model': model3, 'history': hist3, 'best_acc': acc3, 'time': time3}


### MODEL 3: MobileNetV2 ###

Epoch 1/3


Train:   2%|▏         | 3/184 [00:29<29:56,  9.92s/it]


KeyboardInterrupt: 

## 6. Comparison Results

In [10]:
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'Best Validation Accuracy': [results[m]['best_acc'] for m in results.keys()],
    'Training Time (min)': [results[m]['time']/60 for m in results.keys()],
    'Final Train Acc': [results[m]['history']['train_acc'][-1] for m in results.keys()],
    'Final Val Acc': [results[m]['history']['val_acc'][-1] for m in results.keys()]
}).sort_values('Best Validation Accuracy', ascending=False)

print('='*80)
print('3-EPOCH ACCURACY COMPARISON')
print('='*80)
print(comparison.to_string(index=False))
print('='*80)

best = comparison.iloc[0]['Model']
print(f'\n🏆 Best Model (3 epochs): {best}')
print(f'   Accuracy: {comparison.iloc[0]["Best Validation Accuracy"]:.4f}')
print(f'   Time: {comparison.iloc[0]["Training Time (min)"]:.1f} minutes')

3-EPOCH ACCURACY COMPARISON
   Model  Best Validation Accuracy  Training Time (min)  Final Train Acc  Final Val Acc
ResNet18                  0.981879            33.810357         0.963701       0.981879

🏆 Best Model (3 epochs): ResNet18
   Accuracy: 0.9819
   Time: 33.8 minutes


## 7. Accuracy Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, data) in enumerate(results.items()):
    h = data['history']
    epochs = range(1, len(h['train_acc']) + 1)
    
    axes[idx].plot(epochs, h['train_acc'], 'b-o', label='Train', linewidth=2, markersize=8)
    axes[idx].plot(epochs, h['val_acc'], 'r-s', label='Validation', linewidth=2, markersize=8)
    axes[idx].set_title(f'{name}\nBest: {data["best_acc"]:.4f}', fontweight='bold', fontsize=12)
    axes[idx].set_xlabel('Epoch', fontsize=11)
    axes[idx].set_ylabel('Accuracy', fontsize=11)
    axes[idx].legend(fontsize=10)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim([0, 1])

plt.suptitle('3-Epoch Accuracy Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('3epoch_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print('✅ Saved: 3epoch_comparison.png')

## 8. Summary

In [ ]:
print('='*80)
print('QUICK TEST COMPLETE')
print('='*80)
print(f'\nTested: 3 algorithms with {NUM_EPOCHS} epochs each')
print(f'\nResults:')
for name, data in results.items():
    print(f"  {name}: {data['best_acc']:.4f} accuracy in {data['time']/60:.1f} min")
print(f'\nBest: {best} with {comparison.iloc[0]["Best Validation Accuracy"]:.4f} accuracy')
print('\n✅ Your working project (best_model_ResNet18.pth) is NOT affected')
print('='*80)